In [3]:
from datetime import datetime
import json
import sys
import os
import pandas as pd
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/notebooks/exploration.ipynb"))) # two dirname to get to aml_service path 
    
with open(os.path.join(BASE_DIR, 'config', 'thresholds.json')) as f:
    THRESHOLDS = json.load(f)
sys.path.append(BASE_DIR)

from graph.builder import TransactionsGraph


In [4]:
transactions_path = "/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/IBM/amlWORLD/HI-Small_Trans.csv"
df_transactions = pd.read_csv(transactions_path)

In [6]:
cycled_money_path = "/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/output/accounts_cycled_money.csv"
df_cycled_money = pd.read_csv(cycled_money_path)

In [38]:
garg_index_path = "/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/output/accounts_garg_index.csv"
garg_df = pd.read_csv(garg_index_path)

In [7]:
df_transactions["From_Account"] = (
    df_transactions["From Bank"].astype("string")
    .str.cat(df_transactions["Account"].astype("string"), sep="_")
)
df_transactions["To_Account"] = (
    df_transactions["To Bank"].astype("string")
    .str.cat(df_transactions["Account.1"].astype("string"), sep="_")
)

In [9]:
df_transactions.drop(columns=["From Bank", "Account", "To Bank", "Account.1"], inplace=True)

In [10]:
df_transactions.head()

,Timestamp,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From_Account,To_Account
0,2022/09/01 00:20,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0,10_8000EBD30,10_8000EBD30
1,2022/09/01 00:20,0.01,US Dollar,0.01,US Dollar,Cheque,0,3208_8000F4580,1_8000F5340
2,2022/09/01 00:00,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0,3209_8000F4670,3209_8000F4670
3,2022/09/01 00:02,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0,12_8000F5030,12_8000F5030
4,2022/09/01 00:06,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0,10_8000F5200,10_8000F5200


In [21]:
print(df_transactions["Is Laundering"].value_counts())

Is Laundering
0    5073168
1       5177
Name: count, dtype: int64


In [18]:
all_accounts = list(set(df_transactions["From_Account"]).union(set(df_transactions["To_Account"])))
laundering_transactions = df_transactions[df_transactions["Is Laundering"] == 1]
laundering_accounts = set(laundering_transactions["From_Account"]).union(set(laundering_transactions["To_Account"]))

In [22]:
account_df = pd.DataFrame({
    "Account_ID": all_accounts,
    "is_laundering": [1 if acc in laundering_accounts else 0 for acc in all_accounts]
})

In [23]:
print("Total unique accounts:", len(all_accounts))
print("Total laundering accounts:", len(laundering_accounts))

Total unique accounts: 515088
Total laundering accounts: 6357


In [24]:
graph = TransactionsGraph()
currency_exchange = {
    "US Dollar": 52,
    "Bitcoin": 3314084,
    "Euro": 60,
    "Australian Dollar": 36.5,
    "Yuan": 7.67,
    "Rupee": 0.55,
    "Yen": 0.32,
    "Mexican Peso": 3,
    "UK Pound": 69.75,
    "Ruble": 0.72,
    "Canadian Dollar": 37.19,
    "Swiss Franc": 65.27,
    "Brazil Real": 10.24,
    "Saudi Riyal": 13.84,
    "Shekel": 17.8
}

In [25]:
df_transactions["Amount_Paid_EGP"] = df_transactions.apply(
    lambda row: row["Amount Paid"] * currency_exchange[row["Payment Currency"]],
    axis=1
)
df_transactions["Timestamp"] = pd.to_datetime(df_transactions["Timestamp"])


In [27]:
for index, row in df_transactions.iterrows():
    if index % 10000 == 0:
        print(f"Processing transaction {index} / {len(df_transactions)}")
    graph.add_transaction(from_account=row["From_Account"], to_account=row["To_Account"], amount=row["Amount_Paid_EGP"], timestamp=row["Timestamp"])

Processing transaction 0 / 5078345
Processing transaction 10000 / 5078345
Processing transaction 20000 / 5078345
Processing transaction 30000 / 5078345
Processing transaction 40000 / 5078345
Processing transaction 50000 / 5078345
Processing transaction 60000 / 5078345
Processing transaction 70000 / 5078345
Processing transaction 80000 / 5078345
Processing transaction 90000 / 5078345
Processing transaction 100000 / 5078345
Processing transaction 110000 / 5078345
Processing transaction 120000 / 5078345
Processing transaction 130000 / 5078345
Processing transaction 140000 / 5078345
Processing transaction 150000 / 5078345
Processing transaction 160000 / 5078345
Processing transaction 170000 / 5078345
Processing transaction 180000 / 5078345
Processing transaction 190000 / 5078345
Processing transaction 200000 / 5078345
Processing transaction 210000 / 5078345
Processing transaction 220000 / 5078345
Processing transaction 230000 / 5078345
Processing transaction 240000 / 5078345
Processing tra

In [28]:
df_cycled_money.head()

,Account_ID,Cycled_Money,Input_Money,Output_Money
0,23402_813382840,0.0,68992.56,48972.56
1,45279_8113B3720,0.0,163172.60,0.00
2,317963_806849C10,0.0,0.00,11136.32
3,328151_80A5651F0,0.0,0.00,6458.31
4,310352_806073290,0.0,0.00,4972.80


In [29]:
print(len(df_cycled_money))
print(len(account_df))

515088
515088


In [33]:
df_data = pd.merge(account_df, df_cycled_money, how="left", left_on="Account_ID", right_on="Account_ID")

In [34]:
df_data.head()

,Account_ID,is_laundering,Cycled_Money,Input_Money,Output_Money
0,7042_8035E97F0,0,0.0,13904.80,67862891.72
1,31018_801C7BEB0,0,0.0,0.00,13219.20
2,215064_80B499640,0,0.0,0.00,0.00
3,241121_80FB51D00,0,0.0,1047191.88,0.00
4,355657_814568620,0,0.0,0.00,0.00


In [37]:
df_data["Input_Output_Ratio"] = df_data["Input_Money"] / (df_data["Output_Money"] + 1e-6)
df_data["Cycled_Input_Ratio"] = df_data["Cycled_Money"] / (df_data["Input_Money"] + 1e-6)
df_data["Cycled_Output_Ratio"] = df_data["Cycled_Money"] / (df_data["Output_Money"] + 1e-6)
df_data["indegree"] = df_data["Account_ID"].apply(lambda x: graph.get_indegree(x))
df_data["outdegree"] = df_data["Account_ID"].apply(lambda x: graph.get_outdegree(x))

In [39]:
garg_df.head()

,Account_ID,garg_index
0,145752_810EBADA0,0.249781
1,23833_801DAE230,0.333333
2,18180_80C129240,1.000000
3,3305_800582890,1.000000
4,14_8021606B0,1.000000


In [41]:
df_data = pd.merge(df_data, garg_df[["Account_ID", "garg_index"]], how="left", left_on="Account_ID", right_on="Account_ID")

In [42]:
df_data.head()

,Account_ID,is_laundering,Cycled_Money,Input_Money,Output_Money,Input_Output_Ratio,Cycled_Input_Ratio,Cycled_Output_Ratio,indegree,outdegree,garg_index
0,7042_8035E97F0,0,0.0,13904.80,67862891.72,2.048955e-04,0.0,0.0,1,3,0.333333
1,31018_801C7BEB0,0,0.0,0.00,13219.20,0.000000e+00,0.0,0.0,0,1,0.000000
2,215064_80B499640,0,0.0,0.00,0.00,0.000000e+00,0.0,0.0,0,0,0.000000
3,241121_80FB51D00,0,0.0,1047191.88,0.00,1.047192e+12,0.0,0.0,1,0,1.000000
4,355657_814568620,0,0.0,0.00,0.00,0.000000e+00,0.0,0.0,0,0,0.000000


In [ ]:
df_data["Transactions_Out"] = df_data["Account_ID"].apply(lambda x: graph.graph.out_degree(x)) ## out_degree of graph is number of edges
df_data["Transactions_In"] = df_data["Account_ID"].apply(lambda x: graph.graph.in_degree(x)) ## in_degree of graph is number of edges